In [13]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [14]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st, re
import secrets, smtplib
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write(
    '[theme]\n'
    'base="light"\n'
    'primaryColor="#5268ff"\n'
    'backgroundColor="#f4f8ff"\n'
    'secondaryBackgroundColor="#eaf1ff"\n'
    'textColor="#20375f"\n'
)

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f4f8ff",
    "bg_sidebar": "#ffffff",
    "bg_card": "#ffffff",
    "bg_card_alt": "#eaf1ff",
    "text_main": "#20375f",
    "text_heading": "#17315d",
    "text_muted": "#6c82a7",
    "accent": "#5268ff",
    "accent_hover": "#4055e8",
    "accent_text": "#ffffff",
    "border": "#dce7fb",
    "border_light": "#dce7fb",
    "success": "#10b981",
    "danger": "#ef4444"
}

JWT_SECRET = os.getenv("JWT_SECRET")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
ADMIN_USERNAME = "infosys_admin"
ADMIN_PASSWORD = "infosys@123"
if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Check your Colab Secrets.")
    st.stop()

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button


# Infosys Portal - Cloud Glass UI
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
    --blue: #3478f6;
    --navy: #19345f;
    --muted: #7185a8;
    --border: #d7e4ff;
}

html, body, .stApp {
    font-family: 'Inter', sans-serif !important;
    background:
        radial-gradient(ellipse at 0% 0%, #c4e1ff 0%, transparent 38%),
        radial-gradient(ellipse at 100% 100%, #d4d9ff 0%, transparent 38%),
        linear-gradient(135deg, #f7faff, #edf4ff) !important;
    color: var(--navy) !important;
}

header, footer, div[data-testid="stDecoration"] {
    background: transparent !important;
}

.block-container {
    max-width: 1100px !important;
    padding: 1.5rem 1.5rem 3rem !important;
}

h1, h2, h3, p, label {
    font-family: 'Inter', sans-serif !important;
}

/* Remove the login panel box styling */
div[data-testid="stVerticalBlockBorderWrapper"] {
    background: transparent !important;
    border: none !important;
    border-radius: 0 !important;
    box-shadow: none !important;
    padding: 0 !important;
}

/* Clean white input fields */
div[data-baseweb="input"] {
    background: #ffffff !important;
    border: 1px solid var(--border) !important;
    border-radius: 12px !important;
    box-shadow: none !important;
}

div[data-baseweb="input"]:focus-within {
    border: 1px solid var(--blue) !important;
    box-shadow: 0 0 0 3px rgba(52,120,246,0.10) !important;
}

div[data-baseweb="input"] input {
    background: transparent !important;
    color: var(--navy) !important;
    -webkit-text-fill-color: var(--navy) !important;
}

label p {
    color: #465c80 !important;
    font-weight: 600 !important;
}

/* Blue gradient primary buttons */
div[data-testid="stButton"] button,
div[data-testid="stDownloadButton"] button {
    border: 1px solid #d2dfff !important;
    border-radius: 12px !important;
    min-height: 46px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease !important;
    box-shadow: none !important;
}

div[data-testid="stButton"] button {
    background: #ffffff !important;
    color: #2856a6 !important;
}

div[data-testid="stButton"] button:hover {
    border-color: var(--blue) !important;
    background: #f3f7ff !important;
    color: var(--blue) !important;
}

div[data-testid="stButton"] button[kind="primary"],
div[data-testid="stDownloadButton"] button {
    background: linear-gradient(100deg, #398cff, #5948f5) !important;
    color: #ffffff !important;
    border: none !important;
}

/* Sidebar */
section[data-testid="stSidebar"] {
    background: rgba(255,255,255,0.85) !important;
    border-right: 1px solid #e1eaff !important;
}

/* Dashboard cards */
.pn-card {
    background: rgba(255,255,255,0.88) !important;
    border: 1px solid #e1eaff !important;
    border-radius: 18px !important;
    padding: 24px !important;
    box-shadow: 0 8px 28px rgba(58,104,180,0.08) !important;
}

/* Prevent the form buttons from becoming too narrow */
div[data-testid="stButton"] button {
    width: 100% !important;
    white-space: normal !important;
}
/* Center authentication pages */
.auth-page {
    max-width: 650px;
    margin: 0 auto;
}

/* Keep authentication cards consistent */
.auth-page [data-testid="stVerticalBlockBorderWrapper"] {
    width: 100%;
    box-sizing: border-box;
}
/* ===== Infosys Portal: Light Blue Login Theme ===== */


/* Keep dashboard background white */
body:has(section[data-testid="stSidebar"]) .stApp {
    background: #ffffff !important;
    color: #163768 !important;
}


/* Wider page for the two-panel design */
.block-container {
    max-width: 1250px !important;
    padding: 2rem 2rem 3rem !important;
}

/* Light glass-style cards */
div[data-testid="stVerticalBlockBorderWrapper"] {
    background: rgba(255, 255, 255, 0.88) !important;
    border: 1px solid rgba(255, 255, 255, 0.95) !important;
    border-radius: 20px !important;
    box-shadow: 0 14px 40px rgba(40, 85, 150, 0.13) !important;
    padding: 24px !important;
}

/* Text fields */
div[data-testid="stTextInput"] input {
    background: #ffffff !important;
    color: #163768 !important;
    border: 1px solid #d4e3f8 !important;
    border-radius: 10px !important;
    min-height: 42px !important;
}

div[data-testid="stTextInput"] input:focus {
    border-color: #4385f5 !important;
    box-shadow: 0 0 0 2px rgba(67, 133, 245, 0.12) !important;
}

/* Select boxes */
div[data-testid="stSelectbox"] > div > div {
    background: #ffffff !important;
    border-radius: 10px !important;
    border-color: #d4e3f8 !important;
    color: #163768 !important;
}

/* Primary and secondary buttons */
div[data-testid="stButton"] button {
    width: 100% !important;
    min-height: 42px !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease !important;
}

/* Primary button */
div[data-testid="stButton"] button[kind="primary"] {
    background: linear-gradient(100deg, #4385f5, #2868d9) !important;
    color: #ffffff !important;
    border: none !important;
}

/* Blue gradient for secondary buttons */
div[data-testid="stButton"] button[kind="secondary"] {
    background: linear-gradient(100deg, #4385f5, #2868d9) !important;
    color: #ffffff !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
}

div[data-testid="stButton"] button[kind="secondary"]:hover {
    background: linear-gradient(100deg, #2868d9, #174fae) !important;
    color: #ffffff !important;
    border: none !important;
    box-shadow: 0 4px 14px rgba(40, 104, 217, 0.15) !important;
}

/* Hover effect for other buttons */
div[data-testid="stButton"] button:hover {
    border-color: #4385f5 !important;
}

/* Headings and labels */
h1, h2, h3 {
    color: #15366b !important;
}

label, p, .stMarkdown {
    color: #354f73;
}

/* Responsive layout */
@media (max-width: 768px) {
    .block-container {
        padding: 1rem !important;
    }
}
/* Compact login card without affecting other pages */
div[data-testid="stHorizontalBlock"] > div:nth-child(2)
div[data-testid="stVerticalBlockBorderWrapper"] {
    width: 100% !important;
    max-width: 500px !important;
    margin-left: auto !important;
    margin-right: 0 !important;
    box-sizing: border-box !important;
}

/* Prevent inputs and buttons from overflowing */
div[data-testid="stTextInput"],
div[data-testid="stButton"] {
    max-width: 100% !important;
    box-sizing: border-box !important;
}


/* Forgot Password: fully visible, blue, right-aligned */
div[data-testid="stButton"]:has(
    button[data-testid="stBaseButton-tertiary"]
) {
    display: flex !important;
    justify-content: flex-end !important;
    width: 100% !important;
    overflow: visible !important;
    margin-top: 0 !important;
    margin-bottom: 8px !important;
}

div[data-testid="stButton"] button[data-testid="stBaseButton-tertiary"] {
    width: auto !important;
    min-width: 145px !important;
    max-width: none !important;
    flex-shrink: 0 !important;
    white-space: nowrap !important;
    overflow: visible !important;
    text-overflow: clip !important;
    padding: 2px 0 2px 8px !important;
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    color: #2868c7 !important;
    font-weight: 600 !important;
    text-align: right !important;
}

div[data-testid="stButton"] button[data-testid="stBaseButton-tertiary"]:hover {
    background: transparent !important;
    color: #174fae !important;
    text-decoration: underline !important;
}


/* Keep login inputs completely inside their column */
div[data-testid="stTextInput"],
div[data-testid="stTextInput"] > div,
div[data-testid="stTextInput"] div[data-baseweb="input"],
div[data-testid="stTextInput"] input {
    width: 100% !important;
    max-width: 100% !important;
    min-width: 0 !important;
    box-sizing: border-box !important;
}

/* Prevent password visibility icon from causing overflow */
div[data-testid="stTextInput"] div[data-baseweb="base-input"] {
    width: 100% !important;
    max-width: 100% !important;
    box-sizing: border-box !important;
}

/* Keep login column within its available space */
div[data-testid="stHorizontalBlock"] > div {
    min-width: 0 !important;
    box-sizing: border-box !important;
}
/* Sidebar layout */
section[data-testid="stSidebar"] > div {
    height: 100vh !important;
    overflow-y: auto !important;
}

section[data-testid="stSidebar"] [data-testid="stVerticalBlock"] {
    gap: 0.5rem;
}

/* Keep Logout compact */
section[data-testid="stSidebar"] div[data-testid="stButton"] button {
    min-height: 42px !important;
    margin-bottom: 0 !important;
}
/* Dashboard banner title */
.stApp h1 {
    color: #17315d !important;
}

div[style*="background:#17315d"] h1,
div[style*="background: #17315d"] h1 {
    color: #ffffff !important;
}
/* White background only when the user is logged in */
body:has(section[data-testid="stSidebar"]) .stApp,
body:has(section[data-testid="stSidebar"]) .main,
body:has(section[data-testid="stSidebar"]) .block-container {
    background: #ffffff !important;
    background-image: none !important;
}

/* Preserve the blue gradient on authentication pages */
body:not(:has(section[data-testid="stSidebar"])) .stApp {
    background:
        radial-gradient(ellipse at 0% 0%, #c4e1ff 0%, transparent 38%),
        radial-gradient(ellipse at 100% 100%, #d4d9ff 0%, transparent 38%),
        linear-gradient(135deg, #f7faff, #edf4ff) !important;
}
/* Split background ONLY on the Login page */
body:has(input[aria-label="✉️  Email address"]) .stApp {
    background: linear-gradient(
        110deg,
        #ffffff 0%,
        #ffffff 52%,
        #eaf3ff 52%,
        #b8d8ff 100%
    ) fixed !important;
    color: #163768 !important;
}
</style>
""", unsafe_allow_html=True)




def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

def password_error(password):
    if len(password) < 8:
        return "Password must contain at least 8 characters."
    if not re.search(r"[A-Z]", password):
        return "Password must contain an uppercase letter."
    if not re.search(r"[a-z]", password):
        return "Password must contain a lowercase letter."
    if not re.search(r"[0-9]", password):
        return "Password must contain a number."
    if not re.search(r"[^A-Za-z0-9]", password):
        return "Password must contain a special character."
    return None


def valid_email(email):
    return bool(re.fullmatch(
        r"[^@\s]{2,}@[^@.\s]{2,}(?:\.[^@.\s]{2,})+",
        email.strip()
    ))

OTP_EXPIRY_MINUTES = 5

def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"


def make_otp_token(email, otp):
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": email,
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "exp": now + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])

        if (payload.get("sub") != email or
                payload.get("type") != "password_reset_otp"):
            return False, "Invalid verification token."

        if check_txt(input_otp, payload.get("otp_hash", "")):
            return True, "OTP verified successfully."

        return False, "Incorrect OTP."

    except jwt.ExpiredSignatureError:
        return False, "OTP expired. Please request a new code."
    except jwt.PyJWTError:
        return False, "Invalid verification token."


def send_otp_email(to_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Email credentials are not configured."

    message = MIMEText(
        f"Your password-reset OTP is {otp}.\n"
        f"It expires in {OTP_EXPIRY_MINUTES} minutes."
    )
    message["Subject"] = "Infosys Portal - Password Reset OTP"
    message["From"] = EMAIL_ADDRESS
    message["To"] = to_email

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.starttls()
            server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            server.send_message(message)

        return True, "OTP email sent successfully."

    except Exception:
        return False, "Unable to send OTP. Check your Gmail configuration."


with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()



def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(
        f"""
        <div style="text-align:center;padding:10px 0 20px;">
            <div style="font-size:36px;margin-bottom:8px;">⚡</div>
            <div style="font-size:34px;font-weight:700;color:#1670e8;">
                Infosys<sup style="font-size:12px;">®</sup>
            </div>
            <div style="font-size:13px;color:#7185a8;margin-top:3px;">
                Navigate your tomorrow
            </div>
            <div style="font-size:25px;font-weight:700;color:#19345f;margin-top:24px;">
                {title}
            </div>
            <div style="font-size:14px;color:#7184a6;margin-top:7px;">
                {sub}
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )




# ============================================================
# PAGE ROUTING
# ============================================================

if not st.session_state.token:
    if st.session_state.page not in [
        "Login", "Signup", "Forgot", "Admin Login"
    ]:
        st.session_state.page = "Login"

    # Keep authentication pages in the main page area
    with st.container():

        # ==================== LOGIN ====================

        if st.session_state.page == "Login":

            # Two-column login layout
            brand_col, login_col = st.columns([1.15, 0.85], gap="large")


            with brand_col:
                st.markdown(
                    """
                    <div style="text-align:center; padding:100px 15px 60px;">
                        <div style="margin-bottom:20px;">
                            <svg width="76" height="76" viewBox="0 0 100 100"
                                 xmlns="http://www.w3.org/2000/svg">
                                <path d="M50 82 C25 70 15 45 30 20 C42 35 48 53 50 82Z" fill="#2868c7"/>
                                <path d="M50 82 C75 70 85 45 70 20 C58 35 52 53 50 82Z" fill="#2868c7"/>
                                <path d="M50 82 C32 58 35 32 50 10 C65 32 68 58 50 82Z" fill="#4385f5"/>
                                <path d="M50 82 C15 82 5 62 10 45 C30 49 44 64 50 82Z" fill="#1f4f91"/>
                                <path d="M50 82 C85 82 95 62 90 45 C70 49 56 64 50 82Z" fill="#1f4f91"/>
                            </svg>
                        </div>
                        <h1 style="color:#204f91; font-size:34px; font-weight:800;">
                            INFOSYS PORTAL
                        </h1>
                        <p style="color:#64748b; font-size:16px;">
                            Secure Access | Smart Operations
                        </p>
                    </div>
                    """,
                    unsafe_allow_html=True
                )


            with login_col:

                st.markdown(
                    """
                    <div style="text-align:center; padding:20px 0 12px;">
                        <h2 style="color:#15366b; margin-bottom:8px;">
                            Welcome!
                        </h2>
                        <p style="color:#64748b; margin-top:0;">
                            Sign in to your Infosys Portal account
                        </p>
                    </div>
                    """,
                    unsafe_allow_html=True
                )

                email = st.text_input(
                    "✉️  Email address",
                    placeholder="you@infosys.com",
                    key="login_email"
                ).lower().strip()

                pwd = st.text_input(
                    "🔒  Password",
                    type="password",
                    placeholder="Enter your password",
                    key="login_password"
                )

                forgot_spacer, forgot_col = st.columns([3, 1])

                with forgot_col:
                    if st.button(
                        "Forgot Password?",
                        key="login_forgot",
                        type="tertiary",
                        use_container_width=True
                    ):
                        navigate("Forgot")


                st.markdown(
                    "<div style='height:8px'></div>",
                    unsafe_allow_html=True
                )

                if st.button(
                    "Sign In  →",
                    use_container_width=True,
                    type="primary",
                    key="login_submit"
                ):
                    with get_db() as c:
                        r = c.execute(
                            "SELECT password_hash FROM users WHERE email=?",
                            (email,)
                        ).fetchone()

                    if email == "infosys@ai":
                        st.error(
                            "Please use Administrator Login."
                        )
                    elif r and check_txt(pwd, r[0]):
                        st.session_state.token = make_jwt(email)
                        navigate("Dashboard")
                    else:
                        st.error("Invalid credentials.")

                st.markdown(
                    "<div style='text-align:center;color:#7185a8;"
                    "font-size:13px;margin:8px 0;'>or</div>",
                    unsafe_allow_html=True
                )

                left, right = st.columns(2)

                with left:
                    if st.button(
                        "＋ Create Account",
                        use_container_width=True,
                        key="login_signup"
                    ):
                        navigate("Signup")

                with right:
                    if st.button(
                        "♧ Administrator Login",
                        use_container_width=True,
                        key="login_admin"
                    ):
                        navigate("Admin Login")


        # ==================== ADMIN LOGIN ====================

        elif st.session_state.page == "Admin Login":
            _, admin_col, _ = st.columns([1, 2, 1])

            with admin_col:
                with st.container(border=True):

                    auth_header(
                        "Administrator Login",
                        "Restricted administrator access"
                    )

                    admin_user = st.text_input(
                        "Admin username",
                        key="admin_username"
                    )

                    admin_pass = st.text_input(
                        "Admin password",
                        type="password",
                        key="admin_password"
                    )

                    if st.button(
                        "Admin Sign In →",
                        use_container_width=True
                    ):
                        if (
                            admin_user == ADMIN_USERNAME
                            and secrets.compare_digest(
                                admin_pass,
                                ADMIN_PASSWORD
                            )
                        ):
                            st.session_state.token = make_jwt("infosys@ai")
                            navigate("Dashboard")
                        else:
                            st.error("Invalid administrator credentials.")

                    if st.button(
                        "← Back to Sign In",
                        use_container_width=True
                    ):
                        navigate("Login")


        # ==================== SIGNUP ====================
        elif st.session_state.page == "Signup":
            _, signup_col, _ = st.columns([1, 2, 1])

            with signup_col:

                with st.container(border=True):
                    auth_header(
                        "Create an account",
                        "Join Infosys Portal today"
                    )

                    uname = st.text_input(
                        "Full name / Username",
                        placeholder="Jane Doe"
                    )

                    email = st.text_input(
                        "Email address",
                        placeholder="you@infosys.com"
                    ).lower().strip()

                    pwd = st.text_input(
                        "Password",
                        type="password",
                        placeholder="Min. 8 characters"
                    )

                    confirm_pwd = st.text_input(
                        "Confirm password",
                        type="password",
                        placeholder="Re-enter password"
                    )

                    sq = st.selectbox(
                        "Security Question",
                        [
                            "What is your pet name?",
                            "What is your mother's maiden name?",
                            "What is your favourite city?"
                        ]
                    )

                    sa = st.text_input(
                        "Your answer",
                        placeholder="Security answer"
                    )

                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button(
                        "Create Account & Login →",
                        use_container_width=True
                    ):
                        error = None

                        if not all([
                            uname.strip(),
                            email.strip(),
                            pwd,
                            confirm_pwd,
                            sq,
                            sa.strip()
                        ]):
                            error = "Please fill in all required fields."

                        elif not valid_email(email.strip()):
                            error = "Please enter a valid email address."

                        else:
                            error = password_error(pwd)

                        if not error and pwd != confirm_pwd:
                            error = "Passwords do not match."

                        if error:
                            st.error("⚠️ " + error)

                        else:
                            try:
                                with get_db() as c:
                                    c.execute(
                                        """INSERT INTO users
                                        (username, email, password_hash,
                                         security_question, security_answer_hash)
                                        VALUES (?, ?, ?, ?, ?)""",
                                        (
                                            uname.strip(),
                                            email.strip().lower(),
                                            hash_txt(pwd),
                                            sq,
                                            hash_txt(sa.strip().lower())
                                        )
                                    )

                                st.success(
                                    "Account created successfully. Please sign in."
                                )
                                time.sleep(1)
                                navigate("Login")

                            except sqlite3.IntegrityError:
                                st.error(
                                    "Email or username is already registered."
                                )

                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button(
                        "← Back to Sign In",
                        use_container_width=True
                    ):
                        navigate("Login")

        # ==================== FORGOT PASSWORD ====================
        elif st.session_state.page == "Forgot":
            _, forgot_col, _ = st.columns([1, 2, 1])

            with forgot_col:

                with st.container(border=True):
                    auth_header(
                        "Reset your password",
                        "Choose your verification method"
                    )

                    if not st.session_state.reset_email:
                        email = st.text_input(
                            "Registered email address",
                            placeholder="you@infosys.com",
                            key="reset_email_input"
                        ).strip().lower()

                        col_sq, col_otp = st.columns(2)

                        if col_sq.button(
                            "Via Security Question",
                            use_container_width=True
                        ):
                            with get_db() as c:
                                row = c.execute(
                                    "SELECT security_question FROM users WHERE email = ?",
                                    (email,)
                                ).fetchone()

                            if row:
                                st.session_state.reset_email = email
                                st.session_state.sq_p = row[0]
                                st.session_state.reset_mode = "sq"
                                st.rerun()
                            else:
                                st.error("Email not found.")

                        if col_otp.button(
                            "Via OTP",
                            use_container_width=True
                        ):
                            with get_db() as c:
                                row = c.execute(
                                    "SELECT email FROM users WHERE email = ?",
                                    (email,)
                                ).fetchone()

                            if not row:
                                st.error("Email not found.")

                            elif not EMAIL_ADDRESS or not EMAIL_PASSWORD:
                                st.error("Email credentials are not configured.")

                            else:
                                otp = generate_otp()
                                ok, message = send_otp_email(email, otp)

                                if ok:
                                    st.session_state.reset_email = email
                                    st.session_state.reset_mode = "otp"
                                    st.session_state.otp_token = make_otp_token(
                                        email, otp
                                    )
                                    st.session_state.otp_verified = False

                                    st.success(
                                        "OTP sent. It expires in 5 minutes."
                                    )
                                    st.rerun()
                                else:
                                    st.error(message)

                    elif st.session_state.get("reset_mode") == "sq":
                        st.info(
                            "Security Question: " + st.session_state.sq_p
                        )

                        ans = st.text_input(
                            "Your answer"
                        ).strip().lower()

                        npw = st.text_input(
                            "New password",
                            type="password"
                        )

                        confirm_npw = st.text_input(
                            "Confirm new password",
                            type="password"
                        )

                        if st.button(
                            "Reset Password →",
                            use_container_width=True
                        ):
                            error = password_error(npw)

                            if error:
                                st.error(error)

                            elif npw != confirm_npw:
                                st.error("Passwords do not match.")

                            else:
                                 with get_db() as c:
                                    row = c.execute(
                                        "SELECT security_answer_hash FROM users WHERE email = ?",
                                        (st.session_state.reset_email,)
                                    ).fetchone()

                                    if row and check_txt(ans, row[0]):
                                        cur = c.execute(
                                            "UPDATE users SET password_hash = ? WHERE email = ?",
                                            (
                                                hash_txt(npw),
                                                st.session_state.reset_email
                                            )
                                        )

                                        c.commit()

                                        if cur.rowcount == 1:
                                            st.success(
                                                "Password updated. Please sign in."
                                            )

                                            st.session_state.reset_email = None
                                            st.session_state.reset_mode = None
                                            st.session_state.sq_p = None

                                            navigate("Login")
                                        else:
                                            st.error(
                                                "Password update failed. Please try again."
                                            )

                                    else:
                                        st.error("Incorrect security answer.")

                    elif st.session_state.get("reset_mode") == "otp":
                        if not st.session_state.get("otp_verified", False):
                            otp_input = st.text_input(
                                "Enter your 6-digit OTP",
                                max_chars=6
                            )

                            if st.button(
                                "Verify OTP",
                                use_container_width=True
                            ):
                                ok, message = verify_otp_token(
                                    st.session_state.otp_token,
                                    otp_input.strip(),
                                    st.session_state.reset_email
                                )

                                if ok:
                                    st.session_state.otp_verified = True
                                    st.success(
                                        "OTP verified. Set your new password."
                                    )
                                    st.rerun()
                                else:
                                    st.error(message)

                            if st.button(
                                "Resend OTP",
                                use_container_width=True
                            ):
                                otp = generate_otp()

                                ok, message = send_otp_email(
                                    st.session_state.reset_email,
                                    otp
                                )

                                if ok:
                                    st.session_state.otp_token = make_otp_token(
                                        st.session_state.reset_email,
                                        otp
                                    )
                                    st.session_state.otp_verified = False

                                    st.success("New OTP sent.")
                                    st.rerun()
                                else:
                                    st.error(message)

                        else:
                            npw = st.text_input(
                                "New password",
                                type="password"
                            )

                            confirm_npw = st.text_input(
                                "Confirm new password",
                                type="password"
                            )

                            if st.button(
                                "Update Password →",
                                use_container_width=True
                            ):
                                error = password_error(npw)

                                if error:
                                    st.error(error)

                                elif npw != confirm_npw:
                                    st.error("Passwords do not match.")

                                else:
                                    with get_db() as c:
                                        c.execute(
                                            "UPDATE users SET password_hash = ? WHERE email = ?",
                                            (
                                                hash_txt(npw),
                                                st.session_state.reset_email
                                            )
                                        )

                                    st.success(
                                        "Password updated. Please sign in."
                                    )

                                    st.session_state.reset_email = None
                                    st.session_state.reset_mode = None
                                    st.session_state.otp_token = None
                                    st.session_state.otp_verified = False

                                    navigate("Login")

                    if st.button(
                        "← Cancel",
                        use_container_width=True
                    ):
                        st.session_state.reset_email = None
                        st.session_state.reset_mode = None
                        st.session_state.sq_p = None
                        st.session_state.otp_token = None
                        st.session_state.otp_verified = False

                        navigate("Login")



# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()


    email = payload["email"]

    with get_db() as c:
        user_row = c.execute(
            "SELECT username FROM users WHERE email = ?",
            (email,)
        ).fetchone()

    if not user_row:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    uname = user_row[0]

        # 🚀 SIDEBAR MENU
    with st.sidebar:
        # Infosys branding at the top
        st.markdown(f"""
        <div style="padding:16px 8px 20px;text-align:left;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:800;font-size:18px;color:{COLORS['text_heading']};">
                Infosys Portal
            </div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">
                {"Admin Panel" if email == "infosys@ai" else "Intelligent Analytics"}
            </div>
        </div>
        <hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        # Navigation stays directly below branding
        opts = (
            ["Dashboard", "Settings"]
            if email == "infosys@ai"
            else ["Dashboard", "Analytics", "Reports"]
        )

        icons = (
            ["house", "gear"]
            if email == "infosys@ai"
            else ["house", "graph-up", "file-text"]
        )

        menu = option_menu(
            None,
            opts,
            icons=icons,
            styles={
                "container": {
                    "padding": "0",
                    "background-color": COLORS["bg_sidebar"]
                },
                "nav-link": {
                    "font-size": "15px",
                    "text-align": "left",
                    "margin": "5px 0",
                    "border-radius": "8px",
                    "color": COLORS["text_main"]
                },
                "nav-link-selected": {
                    "background-color": COLORS["accent"],
                    "color": "#ffffff"
                }
            }
        )

        # Space between navigation and bottom account section
        st.markdown(
            '<div style="height:25vh;"></div>',
            unsafe_allow_html=True
        )

        st.markdown(
            f'<hr style="border-color:{COLORS["border_light"]};">',
            unsafe_allow_html=True
        )

        # User details at the bottom
        st.markdown(f"""
        <div style="display:flex;align-items:center;gap:10px;padding:8px 0;">
            <div style="
                width:38px;height:38px;min-width:38px;
                border-radius:50%;background:#e7efff;
                display:flex;align-items:center;justify-content:center;
                color:#2868d9;font-weight:700;">
                {uname[:1].upper()}
            </div>
            <div style="overflow-wrap:anywhere;">
                <div style="font-size:14px;font-weight:700;color:{COLORS['text_heading']};">
                    {uname}
                </div>
                <div style="font-size:11px;color:{COLORS['text_muted']};">
                    {"Administrator" if email == "infosys@ai" else "User account"}
                </div>
            </div>
        </div>
        """, unsafe_allow_html=True)

        if st.button(
            "↪ Logout",
            use_container_width=True,
            key="sidebar_logout",
            type="secondary"
        ):
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("Registered Users")

        with get_db() as c:
            users = c.execute(
                "SELECT username, email FROM users ORDER BY username"
            ).fetchall()

        st.caption(f"Total registered users: {len(users)}")

        st.dataframe(
            [{"Username": user[0], "Email": user[1]} for user in users],
            use_container_width=True,
            hide_index=True
        )


    # 🚀 REGULAR USER DASHBOARD
    else:

        # PAGE 1: DASHBOARD
        if menu == "Dashboard":
            st.title("Welcome back, " + uname + "!")
            st.caption("Here is your system overview.")

            c1, c2, c3, c4 = st.columns(4)

            metrics = [
                (c1, "📄", "Documents Indexed", "128"),
                (c2, "🔍", "Searches Today", "47"),
                (c3, "📊", "Efficiency Score", "98.4%"),
                (c4, "🛡️", "Security Status", "Secured")
            ]

            for col, icon, label, value in metrics:
                col.markdown(f"""
                <div class="pn-card" style="text-align:center;">
                    <div style="font-size:28px;">{icon}</div>
                    <div style="font-size:25px;font-weight:700;
                                color:{COLORS['text_heading']};">
                        {value}
                    </div>
                    <div style="color:{COLORS['text_muted']};
                                font-size:12px;">
                        {label}
                    </div>
                </div>
                """, unsafe_allow_html=True)

            st.subheader("System Health")

            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=92,
                title={"text": "System Health Index"},
                gauge={
                    "axis": {"range": [0, 100]},
                    "bar": {"color": COLORS["accent"]},
                    "bgcolor": COLORS["bg_card_alt"]
                }
            ))

            fig.update_layout(
                paper_bgcolor="rgba(0,0,0,0)",
                font={"color": COLORS["text_main"]},
                height=300
            )
            st.plotly_chart(fig, use_container_width=True)

        # PAGE 2: ANALYTICS
        elif menu == "Analytics":
            st.title("Analytics Center")
            st.caption("Explore sample usage and performance trends.")

            left, right = st.columns(2)

            with left:
                st.subheader("Weekly Activity")

                activity_fig = go.Figure()
                activity_fig.add_trace(go.Bar(
                    x=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
                    y=[24, 38, 31, 45, 52, 29, 40],
                    name="Activities"
                ))
                activity_fig.update_layout(
                    xaxis_title="Day",
                    yaxis_title="Activity count",
                    paper_bgcolor="rgba(0,0,0,0)",
                    plot_bgcolor="rgba(0,0,0,0)",
                    height=350
                )
                st.plotly_chart(activity_fig, use_container_width=True)

            with right:
                st.subheader("Document Categories")

                category_fig = go.Figure(data=[go.Pie(
                    labels=["Reports", "Research", "Other"],
                    values=[45, 35, 20],
                    hole=0.55
                )])
                category_fig.update_layout(
                    paper_bgcolor="rgba(0,0,0,0)",
                    height=350
                )
                st.plotly_chart(category_fig, use_container_width=True)

            st.info(
                "These charts use illustrative sample data. "
                "Connect real application data for actual analytics."
            )

        # PAGE 3: REPORTS
        elif menu == "Reports":
            st.title("Reports Center")
            st.caption("Review and export a sample activity report.")

            report_data = [
                {"Metric": "Documents Indexed", "Value": 128},
                {"Metric": "Searches Today", "Value": 47},
                {"Metric": "Efficiency Score", "Value": "98.4%"},
                {"Metric": "System Health", "Value": "92/100"}
            ]

            st.subheader("Report Preview")
            st.dataframe(
                report_data,
                use_container_width=True,
                hide_index=True
            )

            report_csv = (
                "Metric,Value\n"
                "Documents Indexed,128\n"
                "Searches Today,47\n"
                "Efficiency Score,98.4%\n"
                "System Health,92/100\n"
            )

            st.download_button(
                label="Download Report (CSV)",
                data=report_csv,
                file_name="infosys_activity_report.csv",
                mime="text/csv",
                use_container_width=True
            )

            st.caption(
                "This downloadable report contains illustrative sample data."
            )



Overwriting app.py


In [15]:

import os
import time
import subprocess
from pyngrok import ngrok, conf
from google.colab import userdata

# 1. Retrieve the four required secrets from Colab
SECRET_NAMES = [
    "NGROK_AUTHTOKEN",
    "JWT_SECRET",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD"
]

secrets_config = {
    name: userdata.get(name)
    for name in SECRET_NAMES
}

missing = [
    name for name, value in secrets_config.items()
    if not value
]

if missing:
    raise ValueError(
        "Please add these missing Colab Secrets: "
        + ", ".join(missing)
    )

# 2. Reuse the existing ngrok configuration
ngrok.kill()
!pkill -f streamlit


# 3. Pass the application secrets to Streamlit
env = os.environ.copy()
env["JWT_SECRET"] = secrets_config["JWT_SECRET"]
env["EMAIL_ADDRESS"] = secrets_config["EMAIL_ADDRESS"]
env["EMAIL_PASSWORD"] = secrets_config["EMAIL_PASSWORD"]

# 4. Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)

# 5. Open the ngrok tunnel
time.sleep(3)
public_url = ngrok.connect(8501).public_url

print("=" * 60)
print(f"Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("App is running. Stop this cell to shut it down.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("Ngrok tunnel closed and Streamlit stopped.")


Infosys Portal Live URL: https://motivator-frame-unwitting.ngrok-free.dev
App is running. Stop this cell to shut it down.
Ngrok tunnel closed and Streamlit stopped.
